<a href="https://colab.research.google.com/github/debojit-mukherjee/ai-chatbot/blob/main/chatbot_hugging_face.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# -------------------------------
# 1. Knowledge Base
# -------------------------------
knowledge_base = [
    "Python is a beginner-friendly programming language.",
    "SQL is used to query and manage databases.",
    "APIs let applications talk to each other. REST APIs use GET, POST, PUT, DELETE.",
    "SDLC has phases: Requirements, Design, Implementation, Testing, Deployment, Maintenance."
]

# -------------------------------
# 2. Embeddings + FAISS
# -------------------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(knowledge_base).astype("float32")

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# -------------------------------
# 3. Hugging Face Q&A model
# -------------------------------
try:
    qa_model = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
except Exception as e:
    qa_model = None
    st.warning(f"⚠️ Q&A model not loaded: {e}")

# -------------------------------
# 4. Streamlit App with Contextual Memory
# -------------------------------
st.title("Debojit's AI Chatbot")
st.header("Learn AI step by step")
st.write("Type a question below and click **Get Answer**.")

# Initialize session state for memory
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

question = st.text_input("Ask me a question:")

if st.button("Get Answer"):
    if question.strip() == "":
        st.warning("Please enter a question!")
    else:
        # Encode question
        q_embed = embedder.encode([question]).astype("float32")

        # Retrieve top 3 notes
        distances, indices = index.search(q_embed, k=3)
        retrieved_notes = [knowledge_base[i] for i in indices[0]]

        # Build context from retrieved notes + past dialogue
        past_dialogue = ""
        for chat in st.session_state.chat_history:
            past_dialogue += f"Q: {chat['question']}\nA: {chat['answer']}\n"

        context = past_dialogue + " ".join(retrieved_notes)

        # Generate answer
        if qa_model:
            try:
                response = qa_model(question=question, context=context)
                polished_answer = response['answer']
            except Exception as e:
                polished_answer = f"⚠️ Error generating answer: {e}"
        else:
            polished_answer = "⚠️ Q&A model unavailable."

        # Save to session memory
        st.session_state.chat_history.append({"question": question, "answer": polished_answer})

# Display chat history
if st.session_state.chat_history:
    st.subheader("Conversation History")
    for i, chat in enumerate(st.session_state.chat_history, 1):
        st.write(f"**Q{i}:** {chat['question']}")
        st.write(f"**A{i}:** {chat['answer']}")
        st.write("---")
else:
    st.info("No questions asked yet. Start by typing above!")
